# Explore performance evaluation summary results

In [17]:
import json
from pathlib import Path
import pandas as pd

LOGS_DIR = Path("../logs")
json_files = sorted(list(LOGS_DIR.glob("e1_*best_val_auprc*.json")))

model_rows = []
baseline_all_zero = {}
baseline_prevalence = {}

for file in json_files:
    with open(file, "r") as f:
        content = json.load(f)

    exp_name = content.get("experiment", "")
    if not exp_name.startswith("e1"):
        continue

    metrics = content.get("metrics", {})
    
    # Extract model metrics: raw for global ranking, post-processed for binary detection
    row = {
        "experiment": exp_name,
        "auprc": metrics.get("auprc", {}).get("mean"),
        "precision (Post-PP)": metrics.get("pp_precision", {}).get("mean"),
        "recall (Post-PP)": metrics.get("pp_recall", {}).get("mean"),
        "boundary_f1": metrics.get("boundary_f1", {}).get("mean"),
        "ipsnr": metrics.get("ipsnr", {}).get("mean")
    }
    model_rows.append(row)

    # Extract baseline metrics once
    if not baseline_all_zero:
        b_zero = content.get("baseline_all_zero", {})
        baseline_all_zero = {
            "auprc": b_zero.get("auprc", {}).get("mean"),
            "precision (Post-PP)": b_zero.get("precision", {}).get("mean"),
            "recall (Post-PP)": b_zero.get("recall", {}).get("mean"),
            "boundary_f1": b_zero.get("boundary_f1", {}).get("mean"),
            "ipsnr": None
        }

    if not baseline_prevalence:
        b_prev = content.get("baseline_prevalence_rate", {})
        baseline_prevalence = {
            "auprc": b_prev.get("auprc", {}).get("mean"),
            "precision (Post-PP)": b_prev.get("precision", {}).get("mean"),
            "recall (Post-PP)": b_prev.get("recall", {}).get("mean"),
            "boundary_f1": b_prev.get("boundary_f1", {}).get("mean"),
            "ipsnr": None
        }

COLUMN_ORDER = ["auprc", "precision (Post-PP)", "recall (Post-PP)", "boundary_f1", "ipsnr"]

# Build Model DataFrame
summary_df = pd.DataFrame(model_rows).set_index("experiment").reindex(columns=COLUMN_ORDER)

# Build Baseline DataFrame
baseline_df = pd.DataFrame(
    [baseline_all_zero, baseline_prevalence],
    index=["Baseline: All-Zero", "Baseline: Prevalence-Rate"]
).reindex(columns=COLUMN_ORDER)

# Combine into final table
final_table = pd.concat([summary_df, baseline_df])

print("--- Final Model Evaluation vs Fixed Baselines (Post-PP Precision/Recall) ---")
display(
    final_table.style
    .highlight_max(subset=pd.IndexSlice[summary_df.index, :], axis=0, color="lightgreen")
    .format("{:.6f}", na_rep="N/A")
)

--- Final Model Evaluation vs Fixed Baselines (Post-PP Precision/Recall) ---


,auprc,precision (Post-PP),recall (Post-PP),boundary_f1,ipsnr
e1_haunet_direct_focal_dice,0.010298,0.002128,0.340311,0.000410,0.361759
e1_haunet_direct_focal_dice_dilated,0.016267,0.007190,0.310138,0.000391,0.399669
e1_haunet_direct_focal_dice_no_cov_encoder,0.009568,0.004263,0.460415,0.001409,0.483914
e1_haunet_direct_focal_dice_no_dem,0.004989,0.002045,0.309491,0.001103,1.469630
e1_haunet_direct_focal_dice_no_landcover,0.005365,0.002576,0.504971,0.001452,1.130604
Baseline: All-Zero,0.000618,0.000000,0.000000,0.000000,N/A
Baseline: Prevalence-Rate,0.000618,0.000000,0.000000,0.000000,N/A


In [18]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

output_dir = PROJECT_ROOT / "deliverables" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "evaluation_results.csv"
tex_path = output_dir / "evaluation_table.tex"

# Save CSV
final_table.to_csv(csv_path)

# Format & Save LaTeX snippet
latex_code = final_table.to_latex(
    float_format="{:.6f}".format,
    na_rep="N/A",
    caption="Model performance evaluation across feature ablations vs. fixed baselines.",
    label="tab:evaluation_results",
    column_format="lccccc"
)

with open(tex_path, "w") as f:
    f.write(latex_code)

print(f"Successfully saved to:\n - {csv_path}\n - {tex_path}")

Successfully saved to:
 - /home/jupyter/irp-th25/deliverables/tables/evaluation_results.csv
 - /home/jupyter/irp-th25/deliverables/tables/evaluation_table.tex
